# 03 — Run TEMPTED (R)


In [1]:
suppressPackageStartupMessages({
  library(tempted)
  library(nnet)
})

BATCHES_TO_RUN <- character(0)
RANK <- 5
EXPECTED_BATCHES <- 20
SMOOTH <- 1e-4
MAXITER <- 100
EPSILON <- 1e-4

root <- if (dir.exists("data")) "." else ".."
split_folder <- tail(sort(list.dirs(file.path(root, "data", "splits"), recursive = FALSE)), 1)
output_folder <- file.path(root, "data", "tempted", format(Sys.time(), "%Y%m%d_%H%M%S"))
dir.create(output_folder, recursive = TRUE)

batch_folders <- sort(list.dirs(split_folder, recursive = FALSE))
batch_folders <- batch_folders[grepl("^batch_", basename(batch_folders))]
if (length(BATCHES_TO_RUN)) batch_folders <- batch_folders[basename(batch_folders) %in% BATCHES_TO_RUN]
if (!length(BATCHES_TO_RUN) && length(batch_folders) != EXPECTED_BATCHES) {
  stop(paste("Expected", EXPECTED_BATCHES, "split batches; found", length(batch_folders)))
}
split_run <- basename(split_folder)

subject_table <- function(x, meta) {
  x <- as.data.frame(x)
  names(x) <- paste0("factor_", seq_len(ncol(x)))
  x$subject_id <- rownames(x)
  merge(x, unique(meta[c("subject_id", "label")]), by = "subject_id")
}
balanced_accuracy <- function(y, p) mean(sapply(unique(y), function(x) mean(p[y == x] == x)))
macro_f1 <- function(y, p) mean(sapply(unique(y), function(x) {
  pr <- sum(p == x & y == x) / max(1, sum(p == x))
  re <- sum(p == x & y == x) / max(1, sum(y == x))
  if (pr + re == 0) 0 else 2 * pr * re / (pr + re)
}))


In [2]:
run_batch <- function(folder) {
  started <- proc.time()[3]
  batch <- basename(folder)

  read_matrix <- function(name) {
    x <- read.csv(gzfile(file.path(folder, name)), check.names = FALSE)
    rownames(x) <- as.character(x$sample_id)
    x$sample_id <- NULL
    as.matrix(x)
  }

  train_clr <- read_matrix("train_tempted_clr.csv.gz")
  test_clr <- read_matrix("test_tempted_clr.csv.gz")
  train_meta <- read.csv(gzfile(file.path(folder, "train_metadata.csv.gz")), stringsAsFactors = FALSE)
  test_meta <- read.csv(gzfile(file.path(folder, "test_metadata.csv.gz")), stringsAsFactors = FALSE)
  train_meta$sample_id <- as.character(train_meta$sample_id)
  test_meta$sample_id <- as.character(test_meta$sample_id)
  train_meta$subject_id <- as.character(train_meta$subject_id)
  test_meta$subject_id <- as.character(test_meta$subject_id)

  train_data <- format_tempted(train_clr[train_meta$sample_id, ], train_meta$time, train_meta$subject_id,
                               threshold = 1, transform = "none")
  center <- svd_centralize(train_data, r = 1)
  model <- tempted(center$datlist, r = RANK, smooth = SMOOTH, maxiter = MAXITER, epsilon = EPSILON)

  features <- rownames(train_data[[1]])[-1]
  test_data <- format_tempted(test_clr[test_meta$sample_id, features, drop = FALSE],
                              test_meta$time, test_meta$subject_id, threshold = 1, transform = "none")

  train_scores <- subject_table(model$A_hat, train_meta)
  test_scores <- subject_table(est_test_subject(test_data, model, center), test_meta)
  factors <- grep("^factor_", names(train_scores), value = TRUE)

  means <- sapply(train_scores[factors], mean)
  scales <- sapply(train_scores[factors], sd)
  scales[!is.finite(scales) | scales == 0] <- 1
  train_scores[factors] <- scale(train_scores[factors], means, scales)
  test_scores[factors] <- scale(test_scores[factors], means, scales)

  classifier <- multinom(label ~ ., train_scores[c("label", factors)], trace = FALSE)
  predicted <- as.character(predict(classifier, test_scores[factors]))
  truth <- as.character(test_scores$label)

  predictions <- data.frame(split_run, batch, method = "TEMPTED", subject_id = test_scores$subject_id, truth, predicted)
  metrics <- data.frame(split_run, batch, method = "TEMPTED", status = "success",
                        accuracy = mean(predicted == truth),
                        balanced_accuracy = balanced_accuracy(truth, predicted),
                        macro_f1 = macro_f1(truth, predicted),
                        elapsed_seconds = proc.time()[3] - started, error = "")

  batch_output <- file.path(output_folder, batch)
  dir.create(batch_output)
  B <- as.matrix(model$B_hat)
  if (nrow(B) < ncol(B)) B <- t(B)
  feature_ids <- rownames(B)
  if (is.null(feature_ids)) feature_ids <- features[seq_len(nrow(B))]
  loadings <- data.frame(feature_id = feature_ids, B, check.names = FALSE)
  names(loadings)[-1] <- paste0("factor_", seq_len(ncol(B)))

  write.csv(loadings, gzfile(file.path(batch_output, "feature_loadings.csv.gz")), row.names = FALSE)
  write.csv(train_scores, gzfile(file.path(batch_output, "train_subject_scores.csv.gz")), row.names = FALSE)
  write.csv(test_scores, gzfile(file.path(batch_output, "test_subject_scores.csv.gz")), row.names = FALSE)
  write.csv(predictions, gzfile(file.path(batch_output, "predictions.csv.gz")), row.names = FALSE)
  write.csv(metrics, file.path(batch_output, "metrics.csv"), row.names = FALSE)
  saveRDS(list(model = model, centralization = center), file.path(batch_output, "model.rds"))
  list(metrics = metrics, predictions = predictions)
}

results <- lapply(batch_folders, run_batch)
all_metrics <- do.call(rbind, lapply(results, `[[`, "metrics"))
all_predictions <- do.call(rbind, lapply(results, `[[`, "predictions"))
write.csv(all_metrics, file.path(output_folder, "all_metrics.csv"), row.names = FALSE)
write.csv(all_predictions, gzfile(file.path(output_folder, "all_predictions.csv.gz")), row.names = FALSE)
writeLines(split_run, file.path(output_folder, "source_split.txt"))
cat("Saved:", output_folder, "\nSplit run:", split_run, "\n")
all_metrics


Calculate the 1th Component

Convergence reached at dif=4.15324452202594e-05, iter=3

Calculate the 2th Component

Convergence reached at dif=3.31800257667663e-05, iter=11

Calculate the 3th Component

Convergence reached at dif=8.5041728810604e-05, iter=38

Calculate the 4th Component

Convergence reached at dif=5.61623660273822e-05, iter=7

Calculate the 5th Component

Convergence reached at dif=7.65322781218373e-05, iter=21

Calculate the 1th Component

Convergence reached at dif=1.46257714969022e-05, iter=5

Calculate the 2th Component

Convergence reached at dif=9.96010503324124e-05, iter=10

Calculate the 3th Component

Convergence reached at dif=8.73144003127248e-05, iter=12

Calculate the 4th Component

Convergence reached at dif=6.26646352233943e-05, iter=18

Calculate the 5th Component

Convergence reached at dif=9.22059741961342e-05, iter=64

Calculate the 1th Component

Convergence reached at dif=3.13497575840593e-05, iter=3

Calculate the 2th Component

Convergence reached

Saved: ../data/tempted/20260809_010443 
Split run: 20260809_010356 


,split_run,batch,method,status,accuracy,balanced_accuracy,macro_f1,elapsed_seconds,error
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
elapsed,20260809_010356,batch_001,TEMPTED,success,0.6825397,0.6825397,0.6730463,8.176,
elapsed1,20260809_010356,batch_002,TEMPTED,success,0.6349206,0.6349206,0.6295396,13.132,
elapsed2,20260809_010356,batch_003,TEMPTED,success,0.6507937,0.6507937,0.6513789,12.060,
elapsed3,20260809_010356,batch_004,TEMPTED,success,0.6190476,0.6190476,0.6190476,6.230,
elapsed4,20260809_010356,batch_005,TEMPTED,success,0.6825397,0.6825397,0.6782828,6.964,
elapsed5,20260809_010356,batch_006,TEMPTED,success,0.6190476,0.6190476,0.6103103,6.772,
elapsed6,20260809_010356,batch_007,TEMPTED,success,0.6507937,0.6507937,0.6486761,7.179,
elapsed7,20260809_010356,batch_008,TEMPTED,success,0.6349206,0.6349206,0.6374416,7.186,
elapsed8,20260809_010356,batch_009,TEMPTED,success,0.6666667,0.6666667,0.6629712,5.354,
